In [3]:
from src.models.builders import ModelBuilder
ModelBuilder.list_available()

['classifier',
 'classifier_se',
 'classifier_stm',
 'classifier_attnpl',
 'clf_attnpl_t',
 'lstm',
 'thp_type',
 'thp']

In [29]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from src.utils.mask_utils import TriangularCausalMask, ProbMask
from flash_attn.flash_attn_interface import flash_attn_varlen_qkvpacked_func
from flash_attn.bert_padding import unpad_input, pad_input
from  math import sqrt
from abc import ABC, abstractmethod
from src.utils.registrable import Registrable


class BaseAttention(nn.Module, ABC,Registrable):
    """
    Abstract base class for attention mechanisms.

    Subclasses must implement the `forward` method.

    Expected input shapes:
        query, key, value: [B, L, H, D]
        attn_mask: Optional[Tensor] with shape broadcastable to [B, H, L, S]
        padding_mask: Optional[Tensor] with shape [B, L]

    Returns:
        output: [B, L, H, D]
        attention_weights (optional): [B, H, L, S] or None
    """

    def __init__(self,output_attention=False):
        super().__init__()
        self.output_attention = output_attention

    @abstractmethod
    def forward(
        self,
        q: torch.Tensor,
        k: torch.Tensor,
        v: torch.Tensor,
        attn_mask: torch.Tensor = None,
        padding_mask: torch.Tensor = None
    ) -> tuple[torch.Tensor, torch.Tensor | None]:
        """
        Compute attention.

        Args:
            query: [B, L, H, D]
            key: [B, L, H, D]
            value: [B, L, H, D]
            attn_mask: [B, H, L, S] or [B, L, S] or None
            padding_mask: [B, L] or None

        Returns:
            output: [B, L, H, D]
            attention_weights: [B, H, L, S] or None
        """
        pass


@BaseAttention.register(name="standard")
class StandardAttention(BaseAttention):
    """
    Multi-Head Scaled Dot-Product Attention with mask support.

    Args:
        scale (float): Use 1 / sqrt(d_k) as scale.
        attn_dropout (float): Dropout rate after softmax.
        output_attention (bool): Whether to return attention weights.
    """

    def __init__(self, scale, attn_dropout=0.1, output_attention=True):
        super().__init__(output_attention=output_attention)
        self.scale = scale 
        self.dropout = nn.Dropout(attn_dropout)

    def forward(self, q, k, v, attn_mask=None, padding_mask=None):
        """
        Args:
            q, k, v: [B, L, H, D]
            attn_mask: [B, H, L, S] or [B, L, S] or None
            padding_mask: Optional, not used (reserved for flash-attn compatibility)

        Returns:
            output: [B, L, H, D]
            attn_weights (optional): [B, H, L, S]
        """
        # [B, L, H, D] → [B, H, L, D]
        q = q.transpose(1, 2)
        k = k.transpose(1, 2)
        v = v.transpose(1, 2)

        scores = torch.matmul(q , k.transpose(2, 3))  # [B, H, L, S]
        # print("Scores shape:", scores.shape)
        if attn_mask is not None:
            if attn_mask.dim() == 3:
                attn_mask = attn_mask.unsqueeze(1)  # [B, 1, L, S]
            scores = scores.masked_fill(attn_mask, -1e-9)
        
        print("Scores shape:", scores)
        attn_weights = F.softmax(scores* self.scale, dim=-1)
        attn_weights = self.dropout(attn_weights)

        output = torch.matmul(attn_weights, v)  # [B, H, L, D]
        output = output.transpose(1, 2)  # → [B, L, H, D]

        if self.output_attention:
            return output, attn_weights
        else:
            return output, None

@BaseAttention.register(name="full")
class FullAttention(BaseAttention):
    def __init__(self, mask_flag=True, scale=None, attn_dropout=0.1, output_attention=False):
        super().__init__(output_attention=output_attention)
        self.scale = scale
        self.mask_flag = mask_flag
        self.dropout = nn.Dropout(attn_dropout)
        
    def forward(self, q, k, v, attn_mask,padding_mask=None):
        B, L, H, E = q.shape
        _, S, _, D = v.shape
        scale = self.scale or 1./sqrt(E)

        scores = torch.einsum("blhe,bshe->bhls", q, k)
        # print("Scores shape:", scores.shape)
        if self.mask_flag:
            if attn_mask is None:
                attn_mask = TriangularCausalMask(B, L, device=q.device).mask

            scores.masked_fill_(attn_mask, -1e-9)
        print("Scores shape:", scores)
        attn_weights = self.dropout(torch.softmax(scale * scores, dim=-1))
        output = torch.einsum("bhls,bshd->blhd", attn_weights, v)

        if self.output_attention:
            return (output.contiguous(), attn_weights)
        else:
            return (output.contiguous(), None)

# -------------------------------
# 构造 combined attn_mask（causal + padding）
# -------------------------------
def build_combined_mask(B, H, L, S, device):
    # causal mask: [L, S]
    causal_mask = torch.triu(torch.ones(L, S, device=device), diagonal=1).bool()  # True 表示屏蔽未来
    causal_mask = causal_mask.unsqueeze(0).unsqueeze(0).expand(B, H, L, S)  # [B, H, L, S]

    # 假设 S 的最后两个位置是 padding
    padding_mask = torch.zeros(B, S, dtype=torch.bool, device=device)
    padding_mask[:, -2:] = True  # 最后两个为 padding
    padding_mask = padding_mask.unsqueeze(1).unsqueeze(1)  # [B, 1, 1, S]
    padding_mask = padding_mask.expand(B, H, L, S)

    # 最终 mask：logical or
    final_mask = causal_mask | padding_mask
    return final_mask

# -------------------------------
# 测试函数
# -------------------------------
def test_attention_equivalence_with_padding_and_causal_mask():
    B, L, S, H, D = 2, 4, 4, 2, 8
    scale = 1.0 / sqrt(D)
    device = torch.device("cpu")
    torch.manual_seed(42)

    # 构造输入
    q = torch.randn(B, L, H, D, device=device)
    k = torch.randn(B, S, H, D, device=device)
    v = torch.randn(B, S, H, D, device=device)

    # 构造包含 padding 和 causal 的 attn_mask
    attn_mask = build_combined_mask(B, H, L, S, device)

    # 初始化模块
    standard = StandardAttention(scale=scale, attn_dropout=0.0, output_attention=True)
    ##
    full = FullAttention(mask_flag=True, scale=scale, attn_dropout=0.0, output_attention=True)
    out_std, attn_std = standard(q.clone(), k.clone(), v.clone(), attn_mask.clone())
    out_full, attn_full = full(q.clone(), k.clone(), v.clone(), attn_mask.clone())
    ##
    # full = FullAttention(mask_flag=False, scale=scale, attn_dropout=0.0, output_attention=True)
    # out_std, attn_std = standard(q.clone(), k.clone(), v.clone(), None)
    # out_full, attn_full = full(q.clone(), k.clone(), v.clone(), None)

    # 比较
    print("Standard Output shape:", out_std.shape)
    print("Full Output shape:", out_full.shape)
    print("Output一致:", torch.allclose(out_std, out_full, atol=1e-6))
    print("Attention一致:", torch.allclose(attn_std, attn_full, atol=1e-6))

test_attention_equivalence_with_padding_and_causal_mask()

Scores shape: tensor([[[[ 3.2475e+00, -1.0000e-09, -1.0000e-09, -1.0000e-09],
          [ 6.5869e+00, -2.7608e+00, -1.0000e-09, -1.0000e-09],
          [-6.1588e+00,  2.5118e+00, -1.0000e-09, -1.0000e-09],
          [-2.0790e+00,  1.2847e+00, -1.0000e-09, -1.0000e-09]],

         [[-7.0944e-01, -1.0000e-09, -1.0000e-09, -1.0000e-09],
          [ 2.4674e+00,  5.1145e+00, -1.0000e-09, -1.0000e-09],
          [ 2.4812e+00,  3.2185e+00, -1.0000e-09, -1.0000e-09],
          [ 2.7755e+00,  2.6716e+00, -1.0000e-09, -1.0000e-09]]],


        [[[ 4.4805e+00, -1.0000e-09, -1.0000e-09, -1.0000e-09],
          [-5.8768e-01,  1.3723e+00, -1.0000e-09, -1.0000e-09],
          [-4.8000e+00,  2.3821e+00, -1.0000e-09, -1.0000e-09],
          [-3.5210e+00,  2.9143e-01, -1.0000e-09, -1.0000e-09]],

         [[-3.2184e+00, -1.0000e-09, -1.0000e-09, -1.0000e-09],
          [ 1.2057e-01,  4.0429e+00, -1.0000e-09, -1.0000e-09],
          [ 2.3836e+00, -6.3395e-01, -1.0000e-09, -1.0000e-09],
          [ 1.4165

In [30]:
float('-inf')

-inf